In [1]:
from utils import * 
%load_ext autoreload
%autoreload 2

In [2]:
# forward_reads_path = '/groups/banfield/scratch/projects/environmental/sr/int/betazoid/rifle/sed_csp1_16ft.trimmed.PE.1.fastq.gz'
# reverse_reads_path = '/groups/banfield/scratch/projects/environmental/sr/int/betazoid/rifle/sed_csp1_16ft.trimmed.PE.2.fastq.gz'
# ref_path = '/home/philippar/curation/RifSed_csp1_16ft_4_scaffold_2135/scaffold_1.1.fasta'
# output_path = '/home/philippar/curation/RifSed_csp1_16ft_4_scaffold_2135/scaffold_1.1.bam'
# cmd = f'bbmap.sh pigz=t unpigz=t ambiguous=random minid=0.96 idfilter=0.97 threads=64 out=stdout.sam editfilter=5 in1={forward_reads_path} in2={reverse_reads_path} ref={ref_path} nodisk | shrinksam | sambam > {output_path}'
# print(cmd)

In [5]:
script = '''#!/bin/bash 

#SBATCH --job-name={sample_id}
#SBATCH --output={sample_id}.out
#SBATCH --cpus-per-task={num_threads}

cd {ncbi_output_dir}

mkdir -p {output_dir}
mkdir -p {tmp}
prefetch {srr_id} --max-size {max_size}

fasterq-dump {srr_id} --split-files -e {num_threads} -O {output_dir} --temp {tmp}
mv {fasterq_forward_reads_path} {forward_reads_path}
mv {fasterq_reverse_reads_path} {reverse_reads_path}

sickle pe -f {forward_reads_path} -r {reverse_reads_path} -t sanger -o {trimmed_forward_reads_path} -p {trimmed_reverse_reads_path} -s {singles_reads_path} -q {q} -l {l}
pigz -p {num_threads} {trimmed_forward_reads_path}
pigz -p {num_threads} {trimmed_reverse_reads_path}
'''



In [8]:
forward_reads_paths, reverse_reads_paths = dict(), dict()

SRR_ID = 'SRR6159086'
LOCATION = 'lake_superior_sediment'
BIOTITE_OUTPUT_DIR = '/groups/banfield/scratch/projects/environmental/sr/int/betazoid'
BIOTITE_NCBI_OUTPUT_DIR = os.path.join(BIOTITE_OUTPUT_DIR, 'ncbi') # Directory for the SRA cache thing that prefetch and fasterq-dump use.
BIOTITE_TMP_DIR = os.path.join(BIOTITE_OUTPUT_DIR, 'tmp')

PARAMS = dict()
PARAMS['srr_id'] = SRR_ID
PARAMS['sample_id'] = LOCATION
PARAMS['output_dir'] = BIOTITE_OUTPUT_DIR
PARAMS['ncbi_output_dir'] = BIOTITE_NCBI_OUTPUT_DIR
PARAMS['srr_path'] = os.path.join(BIOTITE_NCBI_OUTPUT_DIR, SRR_ID)
PARAMS['tmp'] = BIOTITE_TMP_DIR
PARAMS['forward_reads_path'] = os.path.join(PARAMS['output_dir'], f'{LOCATION}.PE.1.fastq')
PARAMS['reverse_reads_path'] = os.path.join(PARAMS['output_dir'], f'{LOCATION}.PE.2.fastq')
PARAMS['reads_path'] = os.path.join(PARAMS['output_dir'], f'{LOCATION}.fastq')
PARAMS['singles_reads_path'] = os.path.join(PARAMS['output_dir'], f'singles.fastq')
PARAMS['trimmed_forward_reads_path'] = os.path.join(PARAMS['output_dir'], f'{LOCATION}.trimmed.PE.1.fastq')
PARAMS['trimmed_reverse_reads_path'] = os.path.join(PARAMS['output_dir'], f'{LOCATION}.trimmed.PE.2.fastq')
PARAMS['fasterq_forward_reads_path'] = os.path.join(PARAMS['output_dir'], f'{SRR_ID}_1.fastq')
PARAMS['fasterq_reverse_reads_path'] = os.path.join(PARAMS['output_dir'], f'{SRR_ID}_2.fastq')
PARAMS['fasterq_reads_path'] = os.path.join(PARAMS['output_dir'], f'{SRR_ID}.fastq')
PARAMS['max_size'] = 4000000000
PARAMS['num_threads'] = 16
PARAMS['q'] = 20 
PARAMS['l'] = 50

print(script.format(**PARAMS))

print()
print(PARAMS['forward_reads_path'] + '\t' + PARAMS['reverse_reads_path'])

#!/bin/bash 

#SBATCH --job-name=lake_superior_sediment
#SBATCH --output=lake_superior_sediment.out
#SBATCH --cpus-per-task=16

cd /groups/banfield/scratch/projects/environmental/sr/int/betazoid/ncbi

mkdir -p /groups/banfield/scratch/projects/environmental/sr/int/betazoid
mkdir -p /groups/banfield/scratch/projects/environmental/sr/int/betazoid/tmp
prefetch SRR6159086 --max-size 4000000000

fasterq-dump SRR6159086 --split-files -e 16 -O /groups/banfield/scratch/projects/environmental/sr/int/betazoid --temp /groups/banfield/scratch/projects/environmental/sr/int/betazoid/tmp
mv /groups/banfield/scratch/projects/environmental/sr/int/betazoid/SRR6159086_1.fastq /groups/banfield/scratch/projects/environmental/sr/int/betazoid/lake_superior_sediment.PE.1.fastq
mv /groups/banfield/scratch/projects/environmental/sr/int/betazoid/SRR6159086_2.fastq /groups/banfield/scratch/projects/environmental/sr/int/betazoid/lake_superior_sediment.PE.2.fastq

sickle pe -f /groups/banfield/scratch/projects/envi